In [41]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier,plot_tree
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score,log_loss,balanced_accuracy_score,classification_report
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm
import os
os.chdir("/home/pgcp-ai/MachineLearning/Cases/Kyphosis/")

In [2]:
kp = pd.read_csv("Kyphosis.csv")
kp

,Kyphosis,Age,Number,Start
0,absent,71,3,5
1,absent,158,3,14
2,present,128,4,5
3,absent,2,5,1
4,absent,1,4,15
...,...,...,...,...
76,present,157,3,13
77,absent,26,7,13
78,absent,120,2,13
79,present,42,7,6


In [3]:
X, y = kp.drop("Kyphosis", axis = 1), kp["Kyphosis"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 26)

In [5]:
knn = KNeighborsClassifier(n_neighbors=3)
bagg = BaggingClassifier(random_state = 26, estimator = knn, n_estimators = 15)
bagg.fit(X_train, y_train)
y_pred = bagg.predict(X_test)
accuracy_score(y_test, y_pred)

0.88

In [10]:
Ns = np.arange(1,11)
Ss = np.arange(10,21)
scores = []
for n in tqdm(Ns):
    for s in Ss:
        knn = KNeighborsClassifier(n_neighbors=n)
        bag = BaggingClassifier(estimator=knn,n_estimators=s,random_state=26)
        bag.fit(X_train,y_train)
        y_pred = bag.predict(X_test)
        scores.append([n,s,accuracy_score(y_test,y_pred)])
df_scores = pd.DataFrame(scores,columns=['No. Of Neighbors in Knn Estimator','No. of Samples','Accuracy'])
df_scores.sort_values(['Accuracy','No. Of Neighbors in Knn Estimator'],ascending=[False,True])

100%|███████████████████████████████████████████| 10/10 [00:02<00:00,  3.33it/s]


,No. Of Neighbors in Knn Estimator,No. of Samples,Accuracy
11,2,10,0.88
12,2,11,0.88
13,2,12,0.88
14,2,13,0.88
22,3,10,0.88
...,...,...,...
6,1,16,0.80
7,1,17,0.80
8,1,18,0.80
9,1,19,0.80


In [11]:
bm = KNeighborsClassifier(n_neighbors = 2)
bagg = BaggingClassifier(random_state = 26, estimator = bm, n_estimators=10)
bagg.fit(X_train, y_train)
y_pred_prob = bagg.predict_proba(X_test)
log_loss(y_test, y_pred_prob)

1.654990677757713

In [40]:
Ns = np.arange(1,11)
Ss = np.arange(10,21)
scores = []
for n in tqdm(Ns):
    for s in Ss:
        knn = KNeighborsClassifier(n_neighbors=n)
        bag = BaggingClassifier(estimator=knn,n_estimators=s,random_state=26,n_jobs=-1)
        bag.fit(X_train,y_train)
        y_pred_prob = bag.predict_proba(X_test)
        scores.append([n,s,log_loss(y_test,y_pred_prob)])
df_scores = pd.DataFrame(scores,columns=['No. Of Neighbors in Knn Estimator','No. of Samples','Log Loss'])
df_scores.sort_values('Log Loss')

100%|███████████████████████████████████████████| 10/10 [00:32<00:00,  3.24s/it]


,No. Of Neighbors in Knn Estimator,No. of Samples,Log Loss
56,6,11,0.355660
45,5,11,0.356653
46,5,12,0.359435
55,6,10,0.359559
44,5,10,0.360400
...,...,...,...
21,2,20,1.732370
7,1,17,1.734972
8,1,18,1.747036
9,1,19,1.757645


In [21]:
bm = KNeighborsClassifier(n_neighbors = 6)
bagg = BaggingClassifier(random_state = 26, estimator = bm, n_estimators=11)
bagg.fit(X_train, y_train)
y_pred_prob = bagg.predict_proba(X_test)
y_pred = bagg.predict(X_test)
log_loss(y_test, y_pred_prob)

0.3556599369538048

In [23]:
accuracy_score(y_test, y_pred)

0.84

In [24]:
balanced_accuracy_score(y_test, y_pred)

0.5

In [25]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

      absent       0.84      1.00      0.91        21
     present       0.00      0.00      0.00         4

    accuracy                           0.84        25
   macro avg       0.42      0.50      0.46        25
weighted avg       0.71      0.84      0.77        25



/home/pgcp-ai/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/pgcp-ai/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/pgcp-ai/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [37]:
Cs = np.linspace(0.001,5,20)
estimators = np.arange(10,55,5)
scores = []
for c in tqdm(Cs):
    for e in estimators:
        lr = LogisticRegression(C=c)
        bag = BaggingClassifier(estimator=lr,n_estimators=e,random_state=26)
        bag.fit(X_train,y_train)
        y_pred_prob = bag.predict_proba(X_test)
        y_pred = bag.predict(X_test)
        scores.append([c,e,log_loss(y_test,y_pred_prob),accuracy_score(y_test,y_pred)])
df_scores = pd.DataFrame(scores,columns=['C','No. Of Estimators','Log Loss','Accuracy'])
df_scores.sort_values('Log Loss')

100%|███████████████████████████████████████████| 20/20 [00:51<00:00,  2.59s/it]


,C,No. Of Estimators,Log Loss,Accuracy
171,5.000000,10,0.368334,0.80
162,4.736895,10,0.368366,0.80
153,4.473789,10,0.368404,0.80
144,4.210684,10,0.368445,0.80
135,3.947579,10,0.368492,0.80
...,...,...,...,...
5,0.001000,35,0.439904,0.84
2,0.001000,20,0.440225,0.84
4,0.001000,30,0.440249,0.84
3,0.001000,25,0.440461,0.84


In [58]:
lr = LogisticRegression()
kn = KNeighborsClassifier()
dtc = DecisionTreeClassifier(random_state = 26)
lda = LinearDiscriminantAnalysis()
nb = GaussianNB()
estimators = [lr, kn, dtc, lda, nb]
no_of_samples = [10,15,50,100]
scores = []
for e in tqdm(estimators):
    for n in no_of_samples:
        bagg = BaggingClassifier(random_state=26, estimator = e, n_estimators=n,n_jobs=-1)
        bagg.fit(X_train, y_train)
        y_pred_prob = bagg.predict_proba(X_test)
        y_pred = bagg.predict(X_test)
        scores.append([e,n, log_loss(y_test, y_pred_prob),accuracy_score(y_test, y_pred)])

df_scores = pd.DataFrame(scores, columns = ['Estimator Name','No. of Samples', 'Log Loss','Accuracy'])
df_scores.sort_values('Log Loss', ascending = True)

100%|█████████████████████████████████████████████| 5/5 [00:20<00:00,  4.06s/it]


,Estimator Name,No. of Samples,Log Loss,Accuracy
19,GaussianNB(),100,0.306354,0.84
17,GaussianNB(),15,0.310269,0.84
18,GaussianNB(),50,0.311635,0.84
16,GaussianNB(),10,0.314526,0.84
8,DecisionTreeClassifier(random_state=26),10,0.324968,0.96
9,DecisionTreeClassifier(random_state=26),15,0.346642,0.80
11,DecisionTreeClassifier(random_state=26),100,0.348223,0.96
10,DecisionTreeClassifier(random_state=26),50,0.359636,0.88
4,KNeighborsClassifier(),10,0.360400,0.84
0,LogisticRegression(),10,0.370531,0.80
